# Problem-first Neural Network Simulator

Choose an industry → broad goal → specific problem, then receive an **illustrative** allocation of 1–64 heterogeneous computational modules. These are **not** conventional neurons; the planner does not implement those modules or claim its counts are optimal.


In [ ]:
from neural_network_simulator.problem_planner import CATALOG, plan
from dataclasses import asdict
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


## 1. Browse the two-level hierarchy

Industry → broad goal (*predict, optimize, discover*) → concrete problem.

In [ ]:
for industry, goals in CATALOG.items():
    print(f"\n{industry.upper()}")
    for goal, problems in goals.items():
        print(f"  {goal}: {', '.join(problems)}")


## 2. Change the industry/problem and module budget

The count is heuristic. Swap the industry, broad goal, or problem below.


In [ ]:
industry = "finance"
goal = "predict"
problem = "fraud_detection"
units = 12  # 1 to 64
input_types = ["tabular", "timeseries"]
proposal = plan(industry, goal, problem, units=units, input_types=input_types)
print("Recommended roles:", proposal.unit_counts)
print("GPU-eligible units (not GPUs):", proposal.estimated_gpu_units)
print("Warnings:", *proposal.warnings, sep="\n- ")


## 3. Visualize the recommended computational module mix

In [ ]:
df = pd.DataFrame([{"backend": k, "modules": v} for k, v in proposal.unit_counts.items()])
plt.figure(figsize=(11,4))
sns.barplot(data=df, x="backend", y="modules")
plt.title(f"{industry} / {goal} / {problem} — {units} modules")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Override C++ / CUDA or other allocations

The remaining slots are distributed proportionally among unspecified backends.

In [ ]:
custom = plan(industry, goal, problem, units=16, input_types=input_types, overrides={"cpp": 4, "cuda": 3, "pytorch": 3})
pd.DataFrame(custom.architecture).head(16)


## 5. Compare problems across industries

These differences reflect different *problem classes*, not validated per-industry requirements.

In [ ]:
examples = [("finance","predict","fraud_detection"), ("healthcare","optimize","hospital_staffing"), ("entertainment","discover","audience_segments"), ("science","discover","candidate_equations"), ("technology","optimize","kubernetes_scheduling")]
rows = []
for ind, goal_name, problem_name in examples:
    p = plan(ind, goal_name, problem_name, units=16)
    rows.extend({"case": f"{ind}/{problem_name}", "backend": b, "count": n} for b, n in p.unit_counts.items())
pivot = pd.DataFrame(rows).pivot(index="case", columns="backend", values="count")
plt.figure(figsize=(12,4))
sns.heatmap(pivot, annot=True, cmap="Blues", fmt="d")
plt.tight_layout()
plt.show()


## 6. Export a proposed architecture to JSON

The generated JSON is a *plan*, not deployable Kubernetes infrastructure or a trained network. Before training, implement each role, define appropriate input encoders and a ground-truth evaluation dataset.

In [ ]:
import json
from pathlib import Path
out = Path("problem_plan_example.json")
out.write_text(json.dumps(asdict(proposal), indent=2))
print("Saved:", out)


## 7. Next implementation gates

1. Confirm objective and label/constraint definitions.
2. Select data adapters and normalization/embedding strategies.
3. Replace placeholders with real algorithm modules.
4. Validate against a simple baseline on held-out data; measure accuracy, calibration and p95 latency.
5. Profile CPU/GPU transfer, memory and concurrency before estimating GPUs.
6. Only then add Temporal/MLflow lifecycle, CI/CD/CT and Kubernetes resource planning.

High-impact healthcare, finance and civic decisions require additional domain controls and human review.
